In [ ]:
%load_ext autoreload
%autoreload 2

# Preamble

In [ ]:
import sys

sys.path.append(r"..")
from team17ode import *

import colorsys
import numpy as np
from numpy.random import default_rng
import scipy.integrate as itg
import matplotlib as mpl
from matplotlib import pyplot as plt
from mpl_toolkits import mplot3d
from cycler import cycler

In [ ]:
plt = mpl.pyplot
plt.style.use("dark_background")
mpl.rcParams["axes.prop_cycle"] = cycler(color=mpl.color_sequences["Pastel1"])
mpl.rcParams["figure.facecolor"] = "564b3c"
mpl.rcParams["axes.facecolor"] = colorsys.hsv_to_rgb(141.4 / 360, 0.123, 0.447 / 2)
mpl.rcParams["axes.xmargin"] = 0.125
mpl.rcParams["axes.ymargin"] = 0.125
mpl.rcParams["axes3d.xaxis.panecolor"] = (1.0, 0.5, 0.5, 0.0625)
mpl.rcParams["axes3d.yaxis.panecolor"] = (1.0, 0.5, 1.0, 0.0625)
mpl.rcParams["axes3d.zaxis.panecolor"] = (0.0, 0.5, 1.0, 0.0625)
mpl.rcParams["grid.color"] = (1.0, 1.0, 1.0, 0.125)
mpl.rcParams["figure.figsize"] = [9.6, 7.2]
mpl.rcParams["figure.dpi"] = 150
mpl.rcParams["text.usetex"] = True

# Scalar ODE for eigenvalues

Bernoulli differential equation: $y' = y - y^3.$ If $y_0 \in {0, \pm 1}$ then $y = y_0.$ Otherwise substitute $v := y^{-2},$ get
$$v' = -2y^{-3} y' = -2 y^{-3} (y - y^3) = 2(1-v).$$
Then
$$1-v = 1-y^{-2} = C\exp(-2t)$$
so
$$y = \pm(1-C\exp(-2t))^{-1/2}.$$
In terms of initial conditions
$$C = 1-y_0^{-2}.$$
Finally
$$y = \frac{\operatorname{sgn} y_0}{\sqrt{1-(1-y_0^{-2})\exp(-2t)}} = \frac{y_0}{\sqrt{1 + (1-y_0^2)(\exp(-2t)-1)}} := \frac{y_0}{\sqrt{\Delta(t, y_0)}}.$$
This expression is valid for $t > t_* = \frac{1}{2} \log \left( 1 - y_0^{-2} \right)$ when $|y_0| > 1.$ If $|y_0| < 1,$ it is valid for all $t.$ This is equivalent to checking if the determinant $\Delta(t, y_0) > 0.$ Since ODE is autonomous, all solutions for various $t_0$ can be gotten by simple shifts.

# Naive RK and Euler

The first method is explicit Euler, and we will take advantage of `scipy`'s existing infrastructure for solving IVPs. Butcher tableau notation from the Hairer, Nørsett, Wanner book:
$$
\def\arraystretch{1.5}\begin{array}{c|c}
   C & A \\ \hline
     & B^T \\
\end{array} =
\def\arraystretch{1.0}\begin{array}{c|c}
   0      &  \\
   c_2    & a_{21} \\
   \vdots & \vdots & \ddots \\
   c_S    & a_{S1}   & \cdots & a_{S,S-1} \\
   \hline
          & b_1 & \cdots & b_{S-1} & b_S \\
\end{array}
$$
where for each stage $s = 1, \ldots, S,$ we find entry of $K = (k_1 \quad \cdots \quad k_S \quad k_{S+1})^T,$
$$
k_s = f\left(t_0 + c_s h\ ,\ y_0 + h \sum_{i=1}^{s-1} a_{si} k_i \right),
$$
and the result of the step is $y_1 = y_0 + h B^T K.$ The next slope $k_{S+1} = f(t_0 + h, y_1)$ is also computed.

# Parameter choice

Choose an initial value:

In [ ]:
t_bounds = (0.0, 2.0)
ivp_spec = random_pff_ivp(default_rng(0x666666), out_dim=4, in_dim=3)

# Fun lil' transformation sequence

In [ ]:
fig: plt.Figure = plt.figure(figsize=14.4 * np.array([1.0, 3.0 / 9]))
axd = fig.subplot_mosaic(
    """
AA..CC....
AABBCCDD.E
..BB..DD.E
""",
    width_ratios=(1, 1, 1, 1, 1, 1, 1, 1, 0.125, 0.125),
    subplot_kw=dict(
        axes_class=mplot3d.Axes3D,
        facecolor="none",
    ),
    per_subplot_kw=dict(E=dict(axes_class=None)),
)

rng = np.random.default_rng(0xABAD5EED)
samples = np.array(
    [
        np.array([1.0 if rng.integers(0, 2) else -1.0, *rng.uniform(-1.0, 1.0, 2)])[
            rng.permutation(3)
        ]
        for _ in range(666)
    ]
)
cmap = mpl.colormaps["BrBG"]

for ax_name in "ABCD":
    ax = axd[ax_name]
    ax.set_proj_type("ortho")
    ax.set_aspect("equal")
    ax.set_xlim(-2.0, 2.0)
    ax.set_ylim(-2.0, 2.0)
    ax.set_zlim(-2.0, 2.0)

points = samples.T
size = np.exp2(-1)
axd["A"].scatter(*points[:3], s=size, color=cmap(0.5))
points = ivp_spec.V.T @ points
axd["B"].scatter(*points[:3], s=size, color=cmap(0.5))
points = ivp_spec.Sigma0 @ points
axd["C"].scatter(*points[:3], s=size, c=cmap(0.5 + 0.25 * points[3]))
points = ivp_spec.U @ points
scatter = axd["D"].scatter(*points[:3], s=size, c=cmap(0.5 + 0.25 * points[3]))
norm = plt.Normalize(-2.0, 2.0)
fig.colorbar(mpl.cm.ScalarMappable(norm=norm, cmap=cmap), axd["E"])

plt.show()

# Graph of Flow Lines and Euler Polygons

First a graph of flow lines alongside a slope field strip

In [ ]:
make_sv_streamplot_fig()
plt.show()

# Doing a bit more error analysis

Do an error analysis and put it in a data structure

In [ ]:
runs_per_method = 6
fixed_step_kwargs_list = [
    dict(
        fun=ivp_spec.pff_fun,
        t_span=t_bounds,
        y0=np.ravel(ivp_spec.X0),
        dense_output=True,
        num_steps=(1 << (2 * i)) * 4,
    )
    for i in range(runs_per_method)
]
adaptive_kwargs_list = [
    dict(
        fun=ivp_spec.pff_fun,
        jac=ivp_spec.pff_jac,
        t_span=t_bounds,
        y0=np.ravel(ivp_spec.X0),
        dense_output=True,
        rtol=0.0,
        atol=np.exp2(-6 * i - 4),
    )
    for i in range(runs_per_method)
]

error_analyses = {
    method.__name__: do_error_analysis(
        method, kwargs_list, ivp_spec.pff_exact_sol_fn, ivp_spec.pff_norm_fn
    )
    for method, kwargs_list in [
        *[
            (method, fixed_step_kwargs_list)
            for method in (FixedRK1, FixedRK3, FixedRK5)
        ],
        *[
            (method, adaptive_kwargs_list)
            for method in (RK21, itg.RK23, itg.RK45, itg.Radau, itg.BDF, itg.LSODA)
        ],
    ]
}

(
    naive_rk11_error_analysis_result,
    naive_rk23_error_analysis_result,
    naive_rk45_error_analysis_result,
    rk11_error_analysis_result,
    rk23_error_analysis_result,
    rk45_error_analysis_result,
    radau_error_analysis_result,
    *_,
) = error_analyses.values()

In [ ]:
make_polygon_fig(
    ivp_spec,
    naive_rk11_error_analysis_result.solns[0],
    figsize=plt.figaspect(415 / 1216),
)
plt.show()

# The three projection graphs way instead of that 3D nonsense

already did the analysis

In [ ]:
make_error_analysis_fig(
    ivp_spec, naive_rk11_error_analysis_result, mpl.colormaps["summer"]
)
make_error_analysis_fig(ivp_spec, rk11_error_analysis_result, mpl.colormaps["autumn"])

plt.show()

# Just repeat for RK3(2) for now

In [ ]:
rk23_facecolor = colorsys.hsv_to_rgb(34.6 / 360, 0.302 * 0.75, 0.337 * 0.75)

make_error_analysis_fig(
    ivp_spec,
    naive_rk23_error_analysis_result,
    mpl.colormaps["winter"],
    figkwargs=dict(figsize=7.2 * np.array([1.0, 1.0]), facecolor=rk23_facecolor),
)
make_error_analysis_fig(
    ivp_spec,
    rk23_error_analysis_result,
    mpl.colormaps["spring"],
    figkwargs=dict(figsize=7.2 * np.array([1.0, 1.0]), facecolor=rk23_facecolor),
)

plt.show()

# Try you a Radau

In [ ]:
make_polygon_fig(
    ivp_spec, radau_error_analysis_result.solns[1], figsize=plt.figaspect(415 / 1216)
)
plt.show()

In [ ]:
make_error_analysis_fig(ivp_spec, radau_error_analysis_result, mpl.colormaps["cool"])
plt.show()

# Work-precision

First get effective function evaluation coefficients:

In [ ]:
jev_scale, lu_scale = measure_neff_scales(ivp_spec)

Can now make a work-precision diagram:

In [ ]:
make_work_precision_fig(error_analyses, jev_scale, lu_scale)
plt.show()